# 07 — Weekly Model Evaluation

This notebook evaluates the NFL projection model throughout the 2026 season.

The evaluation system will:

- Preserve a game level history of model predictions and actual results.
- Track weekly and cumulative straight up, ATS, and total performance.
- Measure margin error, total error, probability accuracy, and calibration.
- Use only projections that were saved before games were played.
- Provide diagnostics that can later be used to improve and calibrate the model.

The evaluation process is designed to work across the full season rather than for one specific week.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path("../..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

WEEKLY_PROJECTIONS_DIR = (
    PROJECT_ROOT / "weekly_projections"
)

WEEKLY_OUTPUT_DIR = (
    WEEKLY_PROJECTIONS_DIR / "outputs"
)

EVALUATION_DIR = (
    WEEKLY_PROJECTIONS_DIR / "evaluation"
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SEASON = 2026

print("Weekly outputs:", WEEKLY_OUTPUT_DIR)
print("Evaluation directory:", EVALUATION_DIR)

Weekly outputs: ..\..\weekly_projections\outputs
Evaluation directory: ..\..\weekly_projections\evaluation


In [3]:
def normalize_projection_file(path, week):
    """
    Convert a saved weekly projection file into one consistent format.

    Only information that was saved before the games is used.
    """

    df = pd.read_csv(path).copy()

    if "Away" in df.columns and "Home" in df.columns:
        df["away_team"] = df["Away"].str.strip()
        df["home_team"] = df["Home"].str.strip()

    elif "Matchup" in df.columns:
        matchup = df["Matchup"].str.split(
            "@",
            n=1,
            expand=True
        )

        df["away_team"] = matchup[0].str.strip()
        df["home_team"] = matchup[1].str.strip()

    else:
        raise ValueError(
            f"Cannot identify teams in {path.name}"
        )

    if "Win Probability" in df.columns:
        win_prob_col = "Win Probability"

    elif "Win Prob" in df.columns:
        win_prob_col = "Win Prob"

    else:
        raise ValueError(
            f"Cannot find win probability in {path.name}"
        )

    normalized = pd.DataFrame({
        "season": SEASON,
        "week": week,

        "away_team": df["away_team"],
        "home_team": df["home_team"],

        "projected_score": df["Projected Score"],

        "model_winner": df["SU Pick"],
        "model_win_probability": pd.to_numeric(
            df[win_prob_col],
            errors="coerce"
        ),

                "model_spread": df["Model Spread"],
        "market_spread": df["Market Spread"],
        "ats_pick": df["ATS Pick"],

        "saved_ats_units": pd.to_numeric(
            df["ATS Units"],
            errors="coerce"
        ) if "ATS Units" in df.columns else np.nan,

        "model_total": pd.to_numeric(
            df["Model Total"],
            errors="coerce"
        ),

        "market_total": pd.to_numeric(
            df["Market Total"],
            errors="coerce"
        ),

        "total_pick": df["O/U Pick"],

        "saved_total_units": pd.to_numeric(
            df["Total Units"],
            errors="coerce"
        ) if "Total Units" in df.columns else np.nan
    })

    return normalized

In [4]:
projection_file_map = {}

for week_dir in sorted(
    WEEKLY_OUTPUT_DIR.glob("week_*")
):

    if not week_dir.is_dir():
        continue

    try:
        week = int(
            week_dir.name.split("_")[1]
        )
    except (IndexError, ValueError):
        continue

    projection_file = (
        week_dir
        / f"week_{week:02d}_projections.csv"
    )

    if projection_file.exists():
        projection_file_map[week] = (
            projection_file
        )


print(
    f"Found {len(projection_file_map)} "
    f"weekly projection files."
)

for week, path in projection_file_map.items():
    print(
        f"Week {week}: "
        f"{path.relative_to(PROJECT_ROOT)}"
    )

Found 5 weekly projection files.
Week 1: weekly_projections\outputs\week_01\week_01_projections.csv
Week 2: weekly_projections\outputs\week_02\week_02_projections.csv
Week 3: weekly_projections\outputs\week_03\week_03_projections.csv
Week 4: weekly_projections\outputs\week_04\week_04_projections.csv
Week 5: weekly_projections\outputs\week_05\week_05_projections.csv


In [5]:
projection_history = []

for week, path in projection_file_map.items():

    if not path.exists():
        print(
            f"Skipping Week {week}: "
            f"{path.name} not found."
        )
        continue

    week_df = normalize_projection_file(
        path,
        week
    )

    projection_history.append(
        week_df
    )

    print(
        f"Week {week}: "
        f"{len(week_df)} projections loaded."
    )

projection_history = pd.concat(
    projection_history,
    ignore_index=True
)

print()
print(
    f"Total historical projections: "
    f"{len(projection_history)}"
)

display(
    projection_history.head(10)
)

Week 1: 16 projections loaded.
Week 2: 16 projections loaded.
Week 3: 16 projections loaded.
Week 4: 16 projections loaded.
Week 5: 15 projections loaded.

Total historical projections: 79


,season,week,away_team,home_team,projected_score,model_winner,model_win_probability,model_spread,market_spread,ats_pick,saved_ats_units,model_total,market_total,total_pick,saved_total_units
0,2026,1,NE,SEA,NE 22.2 - SEA 26.4,SEA,0.616743,SEA -4.2,SEA -3.0,SEA -3.0,NaN,48.547018,44.5,OVER,NaN
1,2026,1,SF,LA,SF 23.7 - LA 26.1,LA,0.566983,LA -2.4,LA -3.5,SF +3.5,NaN,49.882402,48.5,OVER,NaN
2,2026,1,ARI,LAC,ARI 19.2 - LAC 25.6,LAC,0.672676,LAC -6.4,LAC -9.5,ARI +9.5,NaN,44.750466,47.5,UNDER,NaN
3,2026,1,ATL,PIT,ATL 20.7 - PIT 24.3,PIT,0.601114,PIT -3.6,PIT -3.5,PIT -3.5,NaN,44.972030,42.5,OVER,NaN
4,2026,1,BAL,IND,BAL 24.4 - IND 23.6,BAL,0.521855,BAL -0.8,BAL -3.5,IND +3.5,NaN,47.964078,47.5,OVER,NaN
5,2026,1,BUF,HOU,BUF 22.9 - HOU 22.7,BUF,0.506822,BUF -0.2,BUF -1.5,HOU +1.5,NaN,45.567917,44.5,OVER,NaN
6,2026,1,CHI,CAR,CHI 23.3 - CAR 20.6,CHI,0.575887,CHI -2.7,CHI -3.0,CAR +3.0,NaN,43.888549,46.5,UNDER,NaN
7,2026,1,CLE,JAX,CLE 16.9 - JAX 24.5,JAX,0.704396,JAX -7.6,JAX -8.5,CLE +8.5,NaN,41.453674,39.5,OVER,NaN
8,2026,1,DAL,NYG,DAL 24.6 - NYG 23.3,DAL,0.536636,DAL -1.3,DAL -3.0,NYG +3.0,NaN,47.960474,48.5,UNDER,NaN
9,2026,1,GB,MIN,GB 23.1 - MIN 22.8,GB,0.508364,GB -0.3,MIN -1.5,GB +1.5,NaN,45.970149,46.5,UNDER,NaN


In [6]:
expected_projection_count = sum(
    pd.read_csv(path).shape[0]
    for path in projection_file_map.values()
)

assert len(projection_history) == expected_projection_count

assert projection_history[
    [
        "away_team",
        "home_team",
        "model_winner",
        "model_win_probability",
        "model_spread",
        "market_spread",
        "model_total",
        "market_total",
        "total_pick"
    ]
].notna().all().all()

print(
    "Projection history passed initial checks."
)

print(
    f"Total frozen projections: "
    f"{len(projection_history)}"
)

display(
    projection_history.groupby("week").agg(
        games=("home_team", "count"),
        avg_model_total=("model_total", "mean"),
        avg_market_total=("market_total", "mean"),
        avg_win_probability=(
            "model_win_probability",
            "mean"
        )
    )
)

Projection history passed initial checks.
Total frozen projections: 79


,games,avg_model_total,avg_market_total,avg_win_probability
week,,,,
1,16,45.772683,45.2500,0.579527
2,16,45.881250,45.2500,0.638750
3,16,45.887500,44.5625,0.613313
4,16,45.831250,45.1250,0.620938
5,15,45.886667,45.1000,0.576000


In [7]:
CURRENT_SCHEDULE_PATH = (
    WEEKLY_DATA_DIR
    / "2026_schedule_current.parquet"
)

results = pd.read_parquet(
    CURRENT_SCHEDULE_PATH
).copy()

completed_results = results[
    results["home_score"].notna()
    & results["away_score"].notna()
].copy()

print(
    f"Completed games available: "
    f"{len(completed_results)}"
)

display(
    completed_results[
        [
            "week",
            "away_team",
            "home_team",
            "away_score",
            "home_score"
        ]
    ].tail(10)
)

Completed games available: 64


,week,away_team,home_team,away_score,home_score
54,4,IND,WAS,30.0,13.0
55,4,JAX,CIN,22.0,17.0
56,4,KC,LV,30.0,27.0
57,4,LAC,SEA,23.0,30.0
58,4,LA,PHI,24.0,20.0
59,4,MIA,MIN,10.0,15.0
60,4,NE,BUF,29.0,26.0
61,4,NYJ,CHI,12.0,23.0
62,4,TEN,BAL,18.0,24.0
63,4,ATL,NO,45.0,24.0


In [8]:
completed_results["actual_home_margin"] = (
    completed_results["home_score"]
    - completed_results["away_score"]
)

completed_results["actual_total"] = (
    completed_results["home_score"]
    + completed_results["away_score"]
)

completed_results["actual_winner"] = np.where(
    completed_results["home_score"]
    > completed_results["away_score"],
    completed_results["home_team"],
    completed_results["away_team"]
)

actual_results = completed_results[
    [
        "game_id",
        "week",
        "away_team",
        "home_team",
        "away_score",
        "home_score",
        "actual_home_margin",
        "actual_total",
        "actual_winner"
    ]
].copy()

display(actual_results.tail(10))

,game_id,week,away_team,home_team,away_score,home_score,actual_home_margin,actual_total,actual_winner
54,2026_04_IND_WAS,4,IND,WAS,30.0,13.0,-17.0,43.0,IND
55,2026_04_JAX_CIN,4,JAX,CIN,22.0,17.0,-5.0,39.0,JAX
56,2026_04_KC_LV,4,KC,LV,30.0,27.0,-3.0,57.0,KC
57,2026_04_LAC_SEA,4,LAC,SEA,23.0,30.0,7.0,53.0,SEA
58,2026_04_LA_PHI,4,LA,PHI,24.0,20.0,-4.0,44.0,LA
59,2026_04_MIA_MIN,4,MIA,MIN,10.0,15.0,5.0,25.0,MIN
60,2026_04_NE_BUF,4,NE,BUF,29.0,26.0,-3.0,55.0,NE
61,2026_04_NYJ_CHI,4,NYJ,CHI,12.0,23.0,11.0,35.0,CHI
62,2026_04_TEN_BAL,4,TEN,BAL,18.0,24.0,6.0,42.0,BAL
63,2026_04_ATL_NO,4,ATL,NO,45.0,24.0,-21.0,69.0,ATL


In [9]:
evaluated_games = projection_history.merge(
    actual_results,
    on=[
        "week",
        "away_team",
        "home_team"
    ],
    how="left",
    validate="one_to_one"
)

evaluated_games["game_completed"] = (
    evaluated_games["home_score"].notna()
    & evaluated_games["away_score"].notna()
)

print(
    "Historical projections:",
    len(evaluated_games)
)

print(
    "Completed and ready to evaluate:",
    evaluated_games["game_completed"].sum()
)

print(
    "Still awaiting results:",
    (~evaluated_games["game_completed"]).sum()
)

Historical projections: 79
Completed and ready to evaluate: 64
Still awaiting results: 15


In [10]:
graded_games = evaluated_games[
    evaluated_games["game_completed"]
].copy()

print(
    f"Grading {len(graded_games)} "
    f"frozen historical projections."
)

display(
    graded_games[
        [
            "week",
            "away_team",
            "home_team",
            "model_winner",
            "actual_winner",
            "away_score",
            "home_score"
        ]
    ].tail(15)
)

Grading 64 frozen historical projections.


,week,away_team,home_team,model_winner,actual_winner,away_score,home_score
49,4,ARI,NYG,ARI,NYG,24.0,36.0
50,4,DAL,HOU,HOU,DAL,34.0,30.0
51,4,DEN,SF,SF,SF,14.0,24.0
52,4,DET,CAR,DET,CAR,26.0,32.0
53,4,GB,TB,GB,GB,17.0,14.0
54,4,IND,WAS,IND,IND,30.0,13.0
55,4,JAX,CIN,JAX,JAX,22.0,17.0
56,4,KC,LV,KC,KC,30.0,27.0
57,4,LAC,SEA,SEA,SEA,23.0,30.0
58,4,LA,PHI,LA,LA,24.0,20.0


In [11]:
graded_games["su_result"] = np.where(
    graded_games["model_winner"]
    == graded_games["actual_winner"],
    "W",
    "L"
)

su_summary = (
    graded_games
    .groupby("week")["su_result"]
    .value_counts()
    .unstack(fill_value=0)
)

for col in ["W", "L"]:
    if col not in su_summary.columns:
        su_summary[col] = 0

su_summary = su_summary[
    ["W", "L"]
]

su_summary["Record"] = (
    su_summary["W"].astype(str)
    + "-"
    + su_summary["L"].astype(str)
)

su_summary["Games"] = (
    su_summary["W"]
    + su_summary["L"]
)

su_summary["SU %"] = (
    su_summary["W"]
    / su_summary["Games"]
)

su_summary["SU %"] = (
    su_summary["SU %"] * 100
).round(1)

display(
    su_summary[
        [
            "Record",
            "W",
            "L",
            "Games",
            "SU %"
        ]
    ]
)

su_result,Record,W,L,Games,SU %
week,,,,,
1,9-7,9,7,16,56.2
2,9-7,9,7,16,56.2
3,13-3,13,3,16,81.2
4,10-6,10,6,16,62.5


In [12]:
def parse_spread(spread_string):
    """
    Convert a displayed spread such as:
        'BUF -5.5'
        'DET +3.0'
        'PK'
    into the team and numeric spread.
    """

    if pd.isna(spread_string):
        return None, np.nan

    spread_string = str(spread_string).strip()

    if spread_string.upper() in ["PK", "PICK", "PICK'EM"]:
        return None, 0.0

    parts = spread_string.split()

    if len(parts) < 2:
        return None, np.nan

    team = parts[0]

    try:
        spread = float(parts[1])
    except ValueError:
        spread = np.nan

    return team, spread

In [13]:
def grade_ats_pick(row):

    team, spread = parse_spread(
        row["ats_pick"]
    )

    if team is None or pd.isna(spread):
        return np.nan

    if team == row["home_team"]:
        team_score = row["home_score"]
        opponent_score = row["away_score"]

    elif team == row["away_team"]:
        team_score = row["away_score"]
        opponent_score = row["home_score"]

    else:
        return np.nan

    adjusted_margin = (
        team_score
        + spread
        - opponent_score
    )

    if adjusted_margin > 0:
        return "W"

    elif adjusted_margin < 0:
        return "L"

    else:
        return "P"


graded_games["ats_result"] = graded_games.apply(
    grade_ats_pick,
    axis=1
)

In [14]:
graded_games["model_spread_value"] = (
    graded_games["model_spread"]
    .apply(lambda x: parse_spread(x)[1])
)

graded_games["market_spread_value"] = (
    graded_games["market_spread"]
    .apply(lambda x: parse_spread(x)[1])
)

graded_games["ats_difference"] = (
    graded_games["model_spread_value"]
    - graded_games["market_spread_value"]
).abs()

graded_games["model_total"] = pd.to_numeric(
    graded_games["model_total"],
    errors="coerce"
)

graded_games["market_total"] = pd.to_numeric(
    graded_games["market_total"],
    errors="coerce"
)

graded_games["total_difference"] = (
    graded_games["model_total"]
    - graded_games["market_total"]
).abs()

In [15]:
def assign_ats_units(edge):
    if pd.isna(edge) or edge < 1.0:
        return 0.0
    elif edge < 2.0:
        return 0.5
    elif edge < 3.0:
        return 1.0
    elif edge < 4.0:
        return 1.5
    else:
        return 2.0


def assign_total_units(edge):
    if pd.isna(edge) or edge < 2.0:
        return 0.0
    elif edge < 3.5:
        return 0.5
    elif edge < 5.0:
        return 1.0
    elif edge < 7.0:
        return 1.5
    else:
        return 2.0


calculated_ats_units = (
    graded_games["ats_difference"]
    .apply(assign_ats_units)
)

calculated_total_units = (
    graded_games["total_difference"]
    .apply(assign_total_units)
)

graded_games["ats_units"] = (
    graded_games["saved_ats_units"]
    .fillna(calculated_ats_units)
)

graded_games["total_units"] = (
    graded_games["saved_total_units"]
    .fillna(calculated_total_units)
)

In [16]:
ats_summary = (
    graded_games
    .groupby("week")["ats_result"]
    .value_counts()
    .unstack(fill_value=0)
)

for col in ["W", "L", "P"]:
    if col not in ats_summary.columns:
        ats_summary[col] = 0

ats_summary = ats_summary[
    ["W", "L", "P"]
]

ats_summary["Record"] = (
    ats_summary["W"].astype(str)
    + "-"
    + ats_summary["L"].astype(str)
    + "-"
    + ats_summary["P"].astype(str)
)

ats_summary["Decisions"] = (
    ats_summary["W"]
    + ats_summary["L"]
)

ats_summary["ATS %"] = np.where(
    ats_summary["Decisions"] > 0,
    (
        ats_summary["W"]
        / ats_summary["Decisions"]
        * 100
    ),
    np.nan
)

ats_summary["ATS %"] = (
    ats_summary["ATS %"]
    .round(1)
)

display(
    ats_summary[
        [
            "Record",
            "W",
            "L",
            "P",
            "ATS %"
        ]
    ]
)

ats_result,Record,W,L,P,ATS %
week,,,,,
1,5-10-1,5,10,1,33.3
2,7-9-0,7,9,0,43.8
3,10-5-1,10,5,1,66.7
4,10-5-1,10,5,1,66.7


In [17]:
def grade_total_pick(row):
    """
    Grade the model's frozen Over/Under pick against
    the market total that existed when the projection
    was saved.
    """

    pick = str(row["total_pick"]).strip().upper()

    actual_total = row["actual_total"]
    market_total = row["market_total"]

    if pd.isna(actual_total) or pd.isna(market_total):
        return np.nan

    if actual_total == market_total:
        return "P"

    if pick == "OVER":
        return "W" if actual_total > market_total else "L"

    if pick == "UNDER":
        return "W" if actual_total < market_total else "L"

    return np.nan


graded_games["total_result"] = graded_games.apply(
    grade_total_pick,
    axis=1
)

In [18]:
WIN_PROFIT_PER_UNIT = 100 / 110


def calculate_unit_profit(result, units):

    if pd.isna(result) or pd.isna(units) or units == 0:
        return 0.0

    if result == "W":
        return units

    elif result == "L":
        return -(units * 1.1)

    elif result == "P":
        return 0.0

    return 0.0


graded_games["ats_unit_profit"] = graded_games.apply(
    lambda row: calculate_unit_profit(
        row["ats_result"],
        row["ats_units"]
    ),
    axis=1
)

graded_games["total_unit_profit"] = graded_games.apply(
    lambda row: calculate_unit_profit(
        row["total_result"],
        row["total_units"]
    ),
    axis=1
)

In [19]:
total_summary = (
    graded_games
    .groupby("week")["total_result"]
    .value_counts()
    .unstack(fill_value=0)
)

for col in ["W", "L", "P"]:
    if col not in total_summary.columns:
        total_summary[col] = 0

total_summary = total_summary[
    ["W", "L", "P"]
]

total_summary["Record"] = (
    total_summary["W"].astype(str)
    + "-"
    + total_summary["L"].astype(str)
    + "-"
    + total_summary["P"].astype(str)
)

total_summary["Decisions"] = (
    total_summary["W"]
    + total_summary["L"]
)

total_summary["O/U %"] = np.where(
    total_summary["Decisions"] > 0,
    (
        total_summary["W"]
        / total_summary["Decisions"]
        * 100
    ),
    np.nan
)

total_summary["O/U %"] = (
    total_summary["O/U %"]
    .round(1)
)

display(
    total_summary[
        [
            "Record",
            "W",
            "L",
            "P",
            "O/U %"
        ]
    ]
)

total_result,Record,W,L,P,O/U %
week,,,,,
1,6-10-0,6,10,0,37.5
2,6-10-0,6,10,0,37.5
3,7-9-0,7,9,0,43.8
4,6-10-0,6,10,0,37.5


In [20]:
def model_spread_to_home_margin(row):
    """
    Convert the displayed model spread into expected
    home-team margin.

    Positive = home team expected to win.
    Negative = away team expected to win.
    """

    team, spread = parse_spread(
        row["model_spread"]
    )

    if team is None or pd.isna(spread):
        return np.nan

    if team == row["home_team"]:
        return -spread

    elif team == row["away_team"]:
        return spread

    return np.nan


graded_games["model_home_margin"] = graded_games.apply(
    model_spread_to_home_margin,
    axis=1
)

In [21]:
graded_games["margin_error"] = (
    graded_games["model_home_margin"]
    - graded_games["actual_home_margin"]
)

graded_games["absolute_margin_error"] = (
    graded_games["margin_error"].abs()
)

graded_games["total_error"] = (
    graded_games["model_total"]
    - graded_games["actual_total"]
)

graded_games["absolute_total_error"] = (
    graded_games["total_error"].abs()
)

In [22]:
graded_games["model_home_win_probability"] = np.where(
    graded_games["model_winner"]
    == graded_games["home_team"],
    graded_games["model_win_probability"],
    1 - graded_games["model_win_probability"]
)

graded_games["home_win_actual"] = (
    graded_games["actual_winner"]
    == graded_games["home_team"]
).astype(int)

graded_games["brier_score"] = (
    graded_games["model_home_win_probability"]
    - graded_games["home_win_actual"]
) ** 2

In [23]:
diagnostic_summary = (
    graded_games
    .groupby("week")
    .agg(
        Games=("home_team", "count"),

        Margin_MAE=(
            "absolute_margin_error",
            "mean"
        ),

        Margin_Bias=(
            "margin_error",
            "mean"
        ),

        Total_MAE=(
            "absolute_total_error",
            "mean"
        ),

        Total_Bias=(
            "total_error",
            "mean"
        ),

        Brier=(
            "brier_score",
            "mean"
        )
    )
    .round(3)
)

display(diagnostic_summary)

,Games,Margin_MAE,Margin_Bias,Total_MAE,Total_Bias,Brier
week,,,,,,
1,16,12.294,1.169,13.079,-3.665,0.227
2,16,12.781,3.444,11.756,5.444,0.249
3,16,8.062,-0.137,12.138,-1.363,0.218
4,16,7.119,1.694,10.756,-1.919,0.217


In [24]:
def summarize_weekly_units(
    df,
    result_col,
    units_col,
    profit_col
):

    rows = []

    for week in sorted(df["week"].unique()):

        week_data = df[
            (df["week"] == week)
            & (df[units_col] > 0)
            & (df[result_col].isin(["W", "L", "P"]))
        ].copy()

        wins = (week_data[result_col] == "W").sum()
        losses = (week_data[result_col] == "L").sum()
        pushes = (week_data[result_col] == "P").sum()

        decisions = wins + losses

        win_pct = (
            wins / decisions * 100
            if decisions > 0
            else np.nan
        )

        units_risked = week_data[units_col].sum()
        net_units = week_data[profit_col].sum()

        roi = (
            net_units / units_risked * 100
            if units_risked > 0
            else np.nan
        )

        rows.append({
            "Week": week,
            "Bets": len(week_data),
            "Record": f"{wins}-{losses}-{pushes}",
            "Win %": round(win_pct, 1),
            "Units Risked": round(units_risked, 2),
            "Net Units": round(net_units, 2),
            "ROI %": round(roi, 1)
        })

    return pd.DataFrame(rows)


ats_weekly_units = summarize_weekly_units(
    graded_games,
    "ats_result",
    "ats_units",
    "ats_unit_profit"
)

total_weekly_units = summarize_weekly_units(
    graded_games,
    "total_result",
    "total_units",
    "total_unit_profit"
)


print("ATS WEEKLY UNIT PERFORMANCE")
display(ats_weekly_units)

print("TOTAL WEEKLY UNIT PERFORMANCE")
display(total_weekly_units)

ATS WEEKLY UNIT PERFORMANCE


,Week,Bets,Record,Win %,Units Risked,Net Units,ROI %
0,1,10,3-6-1,33.3,8.0,-3.00,-37.5
1,2,9,2-7-0,22.2,7.0,-5.60,-80.0
2,3,12,8-4-0,66.7,12.5,3.05,24.4
3,4,8,5-2-1,71.4,7.0,3.90,55.7


TOTAL WEEKLY UNIT PERFORMANCE


,Week,Bets,Record,Win %,Units Risked,Net Units,ROI %
0,1,7,1-6-0,14.3,5.5,-5.0,-90.9
1,2,9,3-6-0,33.3,6.5,-1.9,-29.2
2,3,9,2-7-0,22.2,9.0,-5.7,-63.3
3,4,8,2-6-0,25.0,7.5,-3.0,-40.0


In [25]:
weekly_performance = pd.DataFrame(
    index=sorted(
        graded_games["week"].unique()
    )
)

weekly_performance.index.name = "Week"


weekly_performance["Games"] = (
    graded_games
    .groupby("week")
    .size()
)

weekly_performance["SU Record"] = (
    su_summary["Record"]
)

weekly_performance["SU %"] = (
    su_summary["SU %"]
)

weekly_performance["ATS Record"] = (
    ats_summary["Record"]
)

weekly_performance["ATS %"] = (
    ats_summary["ATS %"]
)


ats_weekly_lookup = (
    ats_weekly_units
    .set_index("Week")
)

weekly_performance["ATS Unit Record"] = (
    ats_weekly_lookup["Record"]
)

weekly_performance["ATS Units Risked"] = (
    ats_weekly_lookup["Units Risked"]
)

weekly_performance["ATS Net Units"] = (
    ats_weekly_lookup["Net Units"]
)

weekly_performance["ATS Unit ROI %"] = (
    ats_weekly_lookup["ROI %"]
)


weekly_performance["O/U Record"] = (
    total_summary["Record"]
)

weekly_performance["O/U %"] = (
    total_summary["O/U %"]
)

total_weekly_lookup = (
    total_weekly_units
    .set_index("Week")
)

weekly_performance["O/U Unit Record"] = (
    total_weekly_lookup["Record"]
)

weekly_performance["O/U Units Risked"] = (
    total_weekly_lookup["Units Risked"]
)

weekly_performance["O/U Net Units"] = (
    total_weekly_lookup["Net Units"]
)

weekly_performance["O/U Unit ROI %"] = (
    total_weekly_lookup["ROI %"]
)


weekly_performance["Margin MAE"] = (
    diagnostic_summary["Margin_MAE"]
)

weekly_performance["Margin Bias"] = (
    diagnostic_summary["Margin_Bias"]
)

weekly_performance["Total MAE"] = (
    diagnostic_summary["Total_MAE"]
)

weekly_performance["Total Bias"] = (
    diagnostic_summary["Total_Bias"]
)

weekly_performance["Brier"] = (
    diagnostic_summary["Brier"]
)

display(weekly_performance)

,Games,SU Record,SU %,ATS Record,ATS %,ATS Unit Record,ATS Units Risked,ATS Net Units,ATS Unit ROI %,O/U Record,O/U %,O/U Unit Record,O/U Units Risked,O/U Net Units,O/U Unit ROI %,Margin MAE,Margin Bias,Total MAE,Total Bias,Brier
Week,,,,,,,,,,,,,,,,,,,,
1,16,9-7,56.2,5-10-1,33.3,3-6-1,8.0,-3.00,-37.5,6-10-0,37.5,1-6-0,5.5,-5.0,-90.9,12.294,1.169,13.079,-3.665,0.227
2,16,9-7,56.2,7-9-0,43.8,2-7-0,7.0,-5.60,-80.0,6-10-0,37.5,3-6-0,6.5,-1.9,-29.2,12.781,3.444,11.756,5.444,0.249
3,16,13-3,81.2,10-5-1,66.7,8-4-0,12.5,3.05,24.4,7-9-0,43.8,2-7-0,9.0,-5.7,-63.3,8.062,-0.137,12.138,-1.363,0.218
4,16,10-6,62.5,10-5-1,66.7,5-2-1,7.0,3.90,55.7,6-10-0,37.5,2-6-0,7.5,-3.0,-40.0,7.119,1.694,10.756,-1.919,0.217


In [26]:
cumulative_rows = []

for week in sorted(graded_games["week"].unique()):

    through_week = graded_games[
        graded_games["week"] <= week
    ].copy()

    su_wins = (
        through_week["su_result"] == "W"
    ).sum()

    su_losses = (
        through_week["su_result"] == "L"
    ).sum()

    su_games = su_wins + su_losses

    su_pct = (
        su_wins / su_games * 100
        if su_games > 0
        else np.nan
    )

    ats_wins = (
        through_week["ats_result"] == "W"
    ).sum()

    ats_losses = (
        through_week["ats_result"] == "L"
    ).sum()

    ats_pushes = (
        through_week["ats_result"] == "P"
    ).sum()

    ats_decisions = ats_wins + ats_losses

    ats_pct = (
        ats_wins / ats_decisions * 100
        if ats_decisions > 0
        else np.nan
    )

    total_wins = (
        through_week["total_result"] == "W"
    ).sum()

    total_losses = (
        through_week["total_result"] == "L"
    ).sum()

    total_pushes = (
        through_week["total_result"] == "P"
    ).sum()

    total_decisions = total_wins + total_losses

    total_pct = (
        total_wins / total_decisions * 100
        if total_decisions > 0
        else np.nan
    )
 
    ats_wagers = through_week[
        (through_week["ats_units"] > 0)
        & (through_week["ats_result"].isin(["W", "L", "P"]))
    ].copy()

    ats_unit_wins = (
        ats_wagers["ats_result"] == "W"
    ).sum()

    ats_unit_losses = (
        ats_wagers["ats_result"] == "L"
    ).sum()

    ats_unit_pushes = (
        ats_wagers["ats_result"] == "P"
    ).sum()

    ats_units_risked = ats_wagers[
        "ats_units"
    ].sum()

    ats_net_units = ats_wagers[
        "ats_unit_profit"
    ].sum()

    ats_unit_roi = (
        ats_net_units / ats_units_risked * 100
        if ats_units_risked > 0
        else np.nan
    )

    total_wagers = through_week[
        (through_week["total_units"] > 0)
        & (through_week["total_result"].isin(["W", "L", "P"]))
    ].copy()

    total_unit_wins = (
        total_wagers["total_result"] == "W"
    ).sum()

    total_unit_losses = (
        total_wagers["total_result"] == "L"
    ).sum()

    total_unit_pushes = (
        total_wagers["total_result"] == "P"
    ).sum()

    total_units_risked = total_wagers[
        "total_units"
    ].sum()

    total_net_units = total_wagers[
        "total_unit_profit"
    ].sum()

    total_unit_roi = (
        total_net_units / total_units_risked * 100
        if total_units_risked > 0
        else np.nan
    )

    cumulative_rows.append({
        "Through Week": week,

        "Games": len(through_week),

        "SU Record":
            f"{su_wins}-{su_losses}",

        "SU %":
            round(su_pct, 1),

        "ATS Record":
            f"{ats_wins}-{ats_losses}-{ats_pushes}",

        "ATS %":
            round(ats_pct, 1),

        "ATS Unit Record":
            f"{ats_unit_wins}-{ats_unit_losses}-{ats_unit_pushes}",

        "ATS Units Risked":
            round(ats_units_risked, 2),

        "ATS Net Units":
            round(ats_net_units, 2),

        "ATS Unit ROI %":
            round(ats_unit_roi, 1),

        "O/U Record":
            f"{total_wins}-{total_losses}-{total_pushes}",

        "O/U %":
            round(total_pct, 1),

        "O/U Unit Record":
            f"{total_unit_wins}-{total_unit_losses}-{total_unit_pushes}",

        "O/U Units Risked":
            round(total_units_risked, 2),

        "O/U Net Units":
            round(total_net_units, 2),

        "O/U Unit ROI %":
            round(total_unit_roi, 1),

        "Margin MAE":
            round(
                through_week[
                    "absolute_margin_error"
                ].mean(),
                3
            ),

        "Margin Bias":
            round(
                through_week[
                    "margin_error"
                ].mean(),
                3
            ),

        "Total MAE":
            round(
                through_week[
                    "absolute_total_error"
                ].mean(),
                3
            ),

        "Total Bias":
            round(
                through_week[
                    "total_error"
                ].mean(),
                3
            ),

        "Brier":
            round(
                through_week[
                    "brier_score"
                ].mean(),
                3
            )
    })


cumulative_performance = pd.DataFrame(
    cumulative_rows
)

display(cumulative_performance)

,Through Week,Games,SU Record,SU %,ATS Record,ATS %,ATS Unit Record,ATS Units Risked,ATS Net Units,ATS Unit ROI %,...,O/U %,O/U Unit Record,O/U Units Risked,O/U Net Units,O/U Unit ROI %,Margin MAE,Margin Bias,Total MAE,Total Bias,Brier
0,1,16,9-7,56.2,5-10-1,33.3,3-6-1,8.0,-3.00,-37.5,...,37.5,1-6-0,5.5,-5.0,-90.9,12.294,1.169,13.079,-3.665,0.227
1,2,32,18-14,56.2,12-19-1,38.7,5-13-1,15.0,-8.60,-57.3,...,37.5,4-12-0,12.0,-6.9,-57.5,12.538,2.306,12.418,0.889,0.238
2,3,48,31-17,64.6,22-24-2,47.8,13-17-1,27.5,-5.55,-20.2,...,39.6,6-19-0,21.0,-12.6,-60.0,11.046,1.492,12.324,0.139,0.232
3,4,64,41-23,64.1,32-29-3,52.5,18-19-2,34.5,-1.65,-4.8,...,39.1,8-25-0,28.5,-15.6,-54.7,10.064,1.542,11.932,-0.376,0.228


In [27]:
def summarize_unit_performance(df, result_col, units_col, profit_col):

    wagers = df[
        (df[units_col] > 0)
        & (df[result_col].isin(["W", "L", "P"]))
    ].copy()

    wins = (wagers[result_col] == "W").sum()
    losses = (wagers[result_col] == "L").sum()
    pushes = (wagers[result_col] == "P").sum()

    decisions = wins + losses

    win_pct = (
        wins / decisions * 100
        if decisions > 0
        else np.nan
    )

    units_risked = wagers[units_col].sum()
    net_units = wagers[profit_col].sum()

    roi = (
        net_units / units_risked * 100
        if units_risked > 0
        else np.nan
    )

    return {
        "W": wins,
        "L": losses,
        "P": pushes,
        "Record": f"{wins}-{losses}-{pushes}",
        "Win %": round(win_pct, 1),
        "Units Risked": round(units_risked, 2),
        "Net Units": round(net_units, 2),
        "ROI %": round(roi, 1)
    }


ats_unit_performance = summarize_unit_performance(
    graded_games,
    "ats_result",
    "ats_units",
    "ats_unit_profit"
)

total_unit_performance = summarize_unit_performance(
    graded_games,
    "total_result",
    "total_units",
    "total_unit_profit"
)


unit_performance_summary = pd.DataFrame(
    [
        {
            "Market": "ATS",
            **ats_unit_performance
        },
        {
            "Market": "Totals",
            **total_unit_performance
        }
    ]
)

unit_performance_summary

,Market,W,L,P,Record,Win %,Units Risked,Net Units,ROI %
0,ATS,18,19,2,18-19-2,48.6,34.5,-1.65,-4.8
1,Totals,8,25,0,8-25-0,24.2,28.5,-15.60,-54.7


In [28]:
def summarize_by_unit_size(
    df,
    result_col,
    units_col,
    profit_col
):

    rows = []

    for unit_size in [0.5, 1.0, 1.5, 2.0]:

        subset = df[
            (df[units_col] == unit_size)
            & (df[result_col].isin(["W", "L", "P"]))
        ].copy()

        wins = (subset[result_col] == "W").sum()
        losses = (subset[result_col] == "L").sum()
        pushes = (subset[result_col] == "P").sum()

        decisions = wins + losses

        win_pct = (
            wins / decisions * 100
            if decisions > 0
            else np.nan
        )

        units_risked = subset[units_col].sum()
        net_units = subset[profit_col].sum()

        roi = (
            net_units / units_risked * 100
            if units_risked > 0
            else np.nan
        )

        rows.append({
            "Units": unit_size,
            "Bets": len(subset),
            "Record": f"{wins}-{losses}-{pushes}",
            "Win %": round(win_pct, 1),
            "Units Risked": round(units_risked, 2),
            "Net Units": round(net_units, 2),
            "ROI %": round(roi, 1)
        })

    return pd.DataFrame(rows)


ats_by_unit = summarize_by_unit_size(
    graded_games,
    "ats_result",
    "ats_units",
    "ats_unit_profit"
)

total_by_unit = summarize_by_unit_size(
    graded_games,
    "total_result",
    "total_units",
    "total_unit_profit"
)

print("ATS PERFORMANCE BY UNIT SIZE")
display(ats_by_unit)

print("TOTAL PERFORMANCE BY UNIT SIZE")
display(total_by_unit)

ATS PERFORMANCE BY UNIT SIZE


,Units,Bets,Record,Win %,Units Risked,Net Units,ROI %
0,0.5,17,8-8-1,50.0,8.5,-0.40,-4.7
1,1.0,16,6-9-1,40.0,16.0,-3.90,-24.4
2,1.5,4,3-1-0,75.0,6.0,2.85,47.5
3,2.0,2,1-1-0,50.0,4.0,-0.20,-5.0


TOTAL PERFORMANCE BY UNIT SIZE


,Units,Bets,Record,Win %,Units Risked,Net Units,ROI %
0,0.5,16,2-14-0,12.5,8.0,-6.7,-83.8
1,1.0,11,5-6-0,45.5,11.0,-1.6,-14.5
2,1.5,5,1-4-0,20.0,7.5,-5.1,-68.0
3,2.0,1,0-1-0,0.0,2.0,-2.2,-110.0


In [29]:
GAME_HISTORY_PATH = (
    EVALUATION_DIR
    / "game_results_history.parquet"
)

WEEKLY_PERFORMANCE_PATH = (
    EVALUATION_DIR
    / "weekly_performance.csv"
)

CUMULATIVE_PERFORMANCE_PATH = (
    EVALUATION_DIR
    / "cumulative_performance.csv"
)

In [30]:
game_results_history = graded_games[
    [
        "season",
        "week",
        "away_team",
        "home_team",

        "projected_score",

        "model_winner",
        "model_win_probability",
        "model_home_win_probability",

        "model_spread",
        "market_spread",
        "ats_pick",
        "ats_difference",
        "ats_units",
        "ats_unit_profit",

        "model_total",
        "market_total",
        "total_pick",
        "total_difference",
        "total_units",
        "total_unit_profit",
        "away_score",
        "home_score",

        "actual_home_margin",
        "actual_total",
        "actual_winner",

        "su_result",
        "ats_result",
        "total_result",

        "model_home_margin",
        "margin_error",
        "absolute_margin_error",

        "total_error",
        "absolute_total_error",

        "brier_score"
    ]
].copy()

game_results_history = game_results_history.sort_values(
    ["week", "away_team", "home_team"]
).reset_index(drop=True)

game_results_history.to_parquet(
    GAME_HISTORY_PATH,
    index=False
)

weekly_performance.to_csv(
    WEEKLY_PERFORMANCE_PATH
)

cumulative_performance.to_csv(
    CUMULATIVE_PERFORMANCE_PATH,
    index=False
)

print("Saved evaluation outputs:")
print(GAME_HISTORY_PATH)
print(WEEKLY_PERFORMANCE_PATH)
print(CUMULATIVE_PERFORMANCE_PATH)

Saved evaluation outputs:
..\..\weekly_projections\evaluation\game_results_history.parquet
..\..\weekly_projections\evaluation\weekly_performance.csv
..\..\weekly_projections\evaluation\cumulative_performance.csv
